# Category 5 — JOINs & Subqueries
### Total Questions: 8

---

1. Find the name of the customers who never placed an order
2. How many rows will come in outputs of Left, Right, Inner and Outer join from two tables having duplicate rows
3. Write a SQL query to find the second highest salary from the table emp using subquery
4. Write a SQL query to find the employee with second highest salary using subquery style
5. Find all employees who earn more than their managers
6. Write a SQL query for cumulative sum of salary of each employee from Jan to July
7. Given Table X and Y with duplicate ids — determine the count of rows for != join types
8. Zomato multi-table user, partner and order analysis

⬆️ Upload sql_practice.db from your computer

In [1]:
# ⬆️ Upload sql_practice.db from your computer
from google.colab import files
uploaded = files.upload()
print("✅ Database uploaded!")

Saving sql_practice.db to sql_practice.db
✅ Database uploaded!


Verify tables loaded

In [2]:
import sqlite3
import pandas as pd
import os

_cwd = os.getcwd()
_db = os.path.join(_cwd, "sql_practice.db")
if not os.path.isfile(_db):
    _db = os.path.join(os.path.dirname(_cwd), "sql_practice.db")
conn = sqlite3.connect(_db)

# Verify tables loaded
pd.read_sql_query("""
SELECT name FROM sqlite_master
WHERE type='table'
ORDER BY name
""", conn)


,name
0,customers
1,department
2,emp
3,employee_salary
4,employees
5,monthly_salary
6,num
7,orders
8,posts
9,products


## 1. Customers Who Never Placed an Order

**Difficulty:** 🟢 Easy

**Subtopic:** LEFT JOIN + NULL Check

---

**Tables: customers + orders**
```
customers
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| customer_id | int     |
| name        | varchar |
+-------------+---------+

orders
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| order_id    | int     |
| customer_id | int     |
| amount      | int     |
+-------------+---------+
```

- `customer_id` in orders links to `customer_id` in customers.
- A customer who never ordered will have **no matching row** in the orders table.

---

**Problem**

Write a SQL query to find the **names of customers who have never placed an order**.

Return the `name` column only.

**Example 1**

**Input**

customers table:
```
+-------------+---------+
| customer_id | name    |
+-------------+---------+
| 1           | Alice   |
| 2           | Bob     |
| 3           | Charlie |
| 4           | David   |
+-------------+---------+
```

orders table:
```
+----------+-------------+--------+
| order_id | customer_id | amount |
+----------+-------------+--------+
| 1        | 1           | 200    |
| 2        | 1           | 150    |
| 3        | 2           | 300    |
+----------+-------------+--------+
```

**Output**
```
+---------+
| name    |
+---------+
| Charlie |
| David   |
+---------+
```

#  Preview Tables

In [3]:
# 👀 Preview Table 1
print("customers table:")
pd.read_sql_query("SELECT * FROM customers", conn)

customers table:


,customer_id,name
0,1,Alice
1,2,Bob
2,3,Charlie
3,4,David
4,5,Emma
5,6,Frank
6,7,Grace
7,8,Henry
8,9,Isla
9,10,Jack


In [4]:
# 👀 Preview Table 2
print("orders table:")
pd.read_sql_query("SELECT * FROM orders", conn)

orders table:


,order_id,customer_id,amount
0,1,1,200
1,2,1,150
2,3,2,300
3,4,2,450
4,5,3,100
5,6,4,550
6,7,4,200
7,8,5,320
8,9,6,410
9,10,6,180


# ✅ Solution

In [5]:
# ✅ Solution
query = """
SELECT c.name
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.customer_id IS NULL
"""

pd.read_sql_query(query, conn)

,name
0,Henry
1,Isla
2,Jack


**Explanation**

- `LEFT JOIN` returns **all rows from customers** even if there is no matching row in orders.
- When a customer has never ordered, all columns from orders will be `NULL`.
- `WHERE o.customer_id IS NULL` keeps only those customers who have **no matching order**.
- Charlie and David have no orders so they appear in the output.
- This is called the **LEFT JOIN Anti-Pattern** — very common in interviews.

## 2. Row Counts for Different JOIN Types on Tables With Duplicate Rows

**Difficulty:** 🔴 Hard

**Subtopic:** JOIN types + Duplicate rows + COUNT

---

**Tables: table_a + table_b**
```
table_a              table_b
+---------+          +---------+
| column  |          | column  |
+---------+          +---------+
| int     |          | int     |
+---------+          +---------+
```

---

**Problem**

Given two tables with duplicate rows:

Left table A:
```
+--------+
| column |
+--------+
| 1      |
| 1      |
| 1      |
| 2      |
| 2      |
| 3      |
| 4      |
| 5      |
+--------+
```

Right table B:
```
+--------+
| column |
+--------+
| 1      |
| 1      |
| 2      |
| 2      |
| 2      |
| 3      |
| 3      |
| 3      |
| 4      |
+--------+
```

How many rows will come in the output of:
- **INNER JOIN**
- **LEFT JOIN**
- **RIGHT JOIN**
- **FULL OUTER JOIN**

**Expected Output**
```
+------------------+-----------+
| join_type        | row_count |
+------------------+-----------+
| INNER JOIN       | 16        |
| LEFT JOIN        | 19        |
| RIGHT JOIN       | 17        |
| FULL OUTER JOIN  | 20        |
+------------------+-----------+
```

✅ INNER JOIN

In [6]:
# ✅ INNER JOIN
print("INNER JOIN:")
pd.read_sql_query("""
SELECT COUNT(*) AS row_count
FROM table_a a
INNER JOIN table_b b ON a.col = b.col
""", conn)

INNER JOIN:


,row_count
0,16


✅ LEFT JOIN

In [7]:
# ✅ LEFT JOIN
print("LEFT JOIN:")
pd.read_sql_query("""
SELECT COUNT(*) AS row_count
FROM table_a a
LEFT JOIN table_b b ON a.col = b.col
""", conn)

LEFT JOIN:


,row_count
0,17


✅ RIGHT JOIN (simulated in SQLite using reversed LEFT JOIN)

In [8]:
# ✅ RIGHT JOIN (simulated in SQLite using reversed LEFT JOIN)
print("RIGHT JOIN:")
pd.read_sql_query("""
SELECT COUNT(*) AS row_count
FROM table_b b
LEFT JOIN table_a a ON a.col = b.col
""", conn)

RIGHT JOIN:


,row_count
0,16


 ✅ FULL OUTER JOIN

In [9]:
# ✅ FULL OUTER JOIN (simulated in SQLite)
print("FULL OUTER JOIN:")
pd.read_sql_query("""
SELECT COUNT(*) AS row_count FROM (
    SELECT a.col FROM table_a a
    LEFT JOIN table_b b ON a.col = b.col
    UNION ALL
    SELECT b.col FROM table_b b
    LEFT JOIN table_a a ON a.col = b.col
    WHERE a.col IS NULL
)
""", conn)

FULL OUTER JOIN:


,row_count
0,17


**Explanation**

- **INNER JOIN** — matches every row in A with every matching row in B.
  - Value 1: 3 rows in A × 2 rows in B = 6
  - Value 2: 2 rows in A × 3 rows in B = 6
  - Value 3: 1 row in A × 3 rows in B = 3
  - Value 4: 1 row in A × 1 row in B = 1
  - Total = **16 rows**

- **LEFT JOIN** — all rows from A + matching rows from B. Unmatched A rows get NULL.
  - Inner matches = 16 + value 5 has no match in B = 1 extra row
  - Total = **17 rows**

- **RIGHT JOIN** — all rows from B + matching rows from A. Unmatched B rows get NULL.
  - Inner matches = 16 + no unmatched B rows here
  - Total = **16 rows**

- **FULL OUTER JOIN** — all rows from both tables.
  - Union of LEFT + RIGHT = 16 + 1 (value 5 unmatched) = **17 rows**

> 💡 SQLite does not support RIGHT JOIN or FULL OUTER JOIN natively — we simulate them using reversed LEFT JOIN and UNION ALL.

## 3. Second Highest Salary Using Subquery

**Difficulty:** 🟡 Medium

**Subtopic:** Subquery / NOT IN

---

**Table: emp**
```
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| salary      | int     |
+-------------+---------+
```

- `id` represents the employee ID.
- `salary` represents the employee salary.

---

**Problem**

Write a SQL query to find the **second highest salary** from the `emp` table using a **subquery**.

**Example 1**

**Input**

emp table:
```
+----+--------+
| id | salary |
+----+--------+
| 1  | 10000  |
| 2  | 15000  |
| 3  | 20000  |
| 4  | 20000  |
| 5  | 30000  |
+----+--------+
```

**Output**
```
+--------+
| salary |
+--------+
| 20000  |
+--------+
```

Preview the table

In [10]:
# 👀 Preview the table
print("emp table:")
pd.read_sql_query("SELECT * FROM emp", conn)

emp table:


,id,salary
0,1,10000
1,2,15000
2,3,20000
3,4,20000
4,5,30000
5,6,30000
6,7,45000
7,8,50000
8,9,50000
9,10,75000


✅ Solution

In [11]:
# ✅ Solution
query = """
SELECT MAX(salary) AS salary
FROM emp
WHERE salary < (
    SELECT MAX(salary)
    FROM emp
)
"""

pd.read_sql_query(query, conn)

,salary
0,50000


**Explanation**

- The **inner subquery** `SELECT MAX(salary) FROM emp` finds the highest salary which is 30000.
- The **outer query** then finds the `MAX(salary)` from all salaries that are **less than 30000**.
- This gives us 20000 — the second highest salary.
- This approach handles duplicates correctly — even though 20000 appears twice it is returned once.
- This is the classic **subquery approach** — compare with Cat 3 Q2 which uses DENSE_RANK.

## 4. Employee With Second Highest Salary (Subquery Style)

**Difficulty:** 🟡 Medium

**Subtopic:** Subquery / NOT IN

---

**Table: employee_salary**
```
+------------------+---------+
| Column Name      | Type    |
+------------------+---------+
| did              | int     |
| name_of_employee | varchar |
| salary           | int     |
+------------------+---------+
```

- `did` represents the department ID.
- `name_of_employee` represents the employee's name.
- `salary` represents the employee's salary.

---

**Problem**

Write a SQL query to return the **name and salary of the employee with the second highest salary** using a subquery.

**Example 1**

**Input**

employee_salary table:
```
+-----+------------------+--------+
| did | name_of_employee | salary |
+-----+------------------+--------+
| 1   | Alice            | 50000  |
| 1   | Bob              | 65000  |
| 1   | Carol            | 60000  |
| 2   | David            | 70000  |
| 2   | Emma             | 72000  |
| 2   | Frank            | 68000  |
| 3   | Anna             | 55000  |
| 3   | George           | 52000  |
| 3   | Henry            | 51000  |
+-----+------------------+--------+
```

**Output**
```
+------------------+--------+
| name_of_employee | salary |
+------------------+--------+
| David            | 70000  |
+------------------+--------+
```

Preview the table

In [12]:
# 👀 Preview the table
print("employee_salary table:")
pd.read_sql_query("SELECT * FROM employee_salary", conn)

employee_salary table:


,did,name_of_employee,salary
0,1,Alice,50000
1,1,Bob,65000
2,1,Carol,60000
3,2,David,70000
4,2,Emma,72000
5,2,Frank,68000
6,3,Anna,55000
7,3,George,52000
8,3,Henry,51000


✅ Solution

In [13]:
# ✅ Solution
query = """
SELECT name_of_employee,
       salary
FROM employee_salary
WHERE salary = (
    SELECT MAX(salary)
    FROM employee_salary
    WHERE salary < (
        SELECT MAX(salary)
        FROM employee_salary
    )
)
"""

pd.read_sql_query(query, conn)

,name_of_employee,salary
0,David,70000


**Explanation**

- The **innermost subquery** finds the maximum salary — 72000.
- The **middle subquery** finds the maximum salary that is less than 72000 — which is 70000.
- The **outer query** returns the employee whose salary equals 70000.
- This is a **nested subquery** — three levels deep.
- Compare with Cat 3 Q3 which solves the same problem using DENSE_RANK window function.

## 5. Employees Who Earn More Than Their Manager

**Difficulty:** 🟡 Medium

**Subtopic:** Self JOIN / Correlated Subquery

---

**Table: employees**
```
+------------+---------+
| Column Name| Type    |
+------------+---------+
| id         | int     |
| name       | varchar |
| salary     | int     |
| dep_id     | int     |
| manager_id | int     |
| hire_date  | date    |
+------------+---------+
```

- `id` represents the employee ID.
- `manager_id` represents the ID of the employee's manager.
- If `manager_id` is NULL the employee has no manager — they are the top level.

---

**Problem**

Write a SQL query to find all **employees who earn more than their manager**.

Return the employee `name` and `salary`.

**Example 1**

**Input**

employees table:
```
+----+-------+--------+--------+------------+------------+
| id | name  | salary | dep_id | manager_id | hire_date  |
+----+-------+--------+--------+------------+------------+
| 1  | John  | 60000  | 1      | NULL       | 2023-01-01 |
| 2  | Alice | 80000  | 1      | 1          | 2023-03-01 |
| 3  | Bob   | 75000  | 2      | 1          | 2023-05-01 |
| 4  | David | 50000  | 2      | 3          | 2024-01-01 |
+----+-------+--------+--------+------------+------------+
```

**Output**
```
+-------+--------+
| name  | salary |
+-------+--------+
| Alice | 80000  |
| Bob   | 75000  |
+-------+--------+
```

 Preview the table

In [14]:
# 👀 Preview the table
print("employees table:")
pd.read_sql_query("SELECT * FROM employees", conn)

employees table:


,id,name,salary,dep_id,manager_id,hire_date,email
0,1,John,60000,1,NaN,2026-01-15,john@gmail.com
1,2,Alice,80000,1,1.0,2026-02-01,alice@gmail.com
2,3,Bob,75000,2,1.0,2026-01-20,john@gmail.com
3,4,David,50000,2,3.0,2026-03-01,bob@gmail.com
4,5,John,45000,3,3.0,2026-02-15,david@gmail.com
5,6,Emma,90000,1,1.0,2025-06-10,emma@gmail.com
6,7,Frank,55000,2,3.0,2025-08-20,frank@gmail.com
7,8,Grace,72000,3,5.0,2026-01-05,grace@gmail.com
8,9,Henry,48000,3,5.0,2025-11-11,alice@gmail.com
9,10,Isla,95000,1,1.0,2025-04-01,isla@gmail.com


✅ Solution

In [15]:
# ✅ Solution
query = """
SELECT e.name,
       e.salary
FROM employees e
JOIN employees m ON e.manager_id = m.id
WHERE e.salary > m.salary
"""

pd.read_sql_query(query, conn)

,name,salary
0,Alice,80000
1,Bob,75000
2,Emma,90000
3,Grace,72000
4,Henry,48000
5,Isla,95000
6,Karen,83000
7,Mia,74000
8,Nathan,66000
9,Paul,88000


**Explanation**

- We join the `employees` table **with itself** using aliases `e` (employee) and `m` (manager).
- `ON e.manager_id = m.id` links each employee to their manager's row.
- `WHERE e.salary > m.salary` keeps only employees who earn **more than their manager**.
- John has no manager (NULL) so he is excluded automatically by the INNER JOIN.
- Alice (80000) > John (60000) ✅ and Bob (75000) > John (60000) ✅
- David (50000) < Bob (75000) ❌ so David is excluded.

## 6. Cumulative Sum of Salary Per Employee From Jan to July

**Difficulty:** 🟡 Medium

**Subtopic:** Window Functions / SUM OVER + ORDER BY Month

---

**Table: monthly_salary**
```
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| emp_id      | int     |
| month       | varchar |
| salary      | int     |
+-------------+---------+
```

- `emp_id` represents the employee ID.
- `month` represents the month name.
- `salary` represents the salary earned in that month.

---

**Problem**

Write a SQL query to calculate the **cumulative sum of salary** for each employee from **January to July**.

Return `emp_id`, `month`, `salary` and `cumulative_salary`.

**Example 1**

**Input**

monthly_salary table:
```
+--------+-------+--------+
| emp_id | month | salary |
+--------+-------+--------+
| 1      | Jan   | 5000   |
| 1      | Feb   | 5000   |
| 1      | Mar   | 5000   |
| 1      | Apr   | 5000   |
| 1      | May   | 5000   |
| 1      | Jun   | 5000   |
| 1      | Jul   | 5000   |
| 2      | Jan   | 7000   |
| 2      | Feb   | 7000   |
| 2      | Mar   | 7000   |
| 2      | Apr   | 7000   |
| 2      | May   | 7000   |
| 2      | Jun   | 7000   |
| 2      | Jul   | 7000   |
+--------+-------+--------+
```

**Output**
```
+--------+-------+--------+-------------------+
| emp_id | month | salary | cumulative_salary |
+--------+-------+--------+-------------------+
| 1      | Jan   | 5000   | 5000              |
| 1      | Feb   | 5000   | 10000             |
| 1      | Mar   | 5000   | 15000             |
| 1      | Apr   | 5000   | 20000             |
| 1      | May   | 5000   | 25000             |
| 1      | Jun   | 5000   | 30000             |
| 1      | Jul   | 5000   | 35000             |
| 2      | Jan   | 7000   | 7000              |
| 2      | Feb   | 7000   | 14000             |
+--------+-------+--------+-------------------+
```

Preview the table

In [16]:
# 👀 Preview the table
print("monthly_salary table:")
pd.read_sql_query("SELECT * FROM monthly_salary", conn)

monthly_salary table:


,emp_id,month,salary
0,1,Jan,5000
1,1,Feb,5000
2,1,Mar,5000
3,1,Apr,5000
4,1,May,5000
5,1,Jun,5000
6,1,Jul,5000
7,2,Jan,7000
8,2,Feb,7000
9,2,Mar,7000


✅ Solution

In [17]:
# ✅ Solution
query = """
SELECT emp_id,
       month,
       salary,
       SUM(salary) OVER (
           PARTITION BY emp_id
           ORDER BY CASE month
               WHEN 'Jan' THEN 1
               WHEN 'Feb' THEN 2
               WHEN 'Mar' THEN 3
               WHEN 'Apr' THEN 4
               WHEN 'May' THEN 5
               WHEN 'Jun' THEN 6
               WHEN 'Jul' THEN 7
           END
       ) AS cumulative_salary
FROM monthly_salary
"""

pd.read_sql_query(query, conn)

,emp_id,month,salary,cumulative_salary
0,1,Jan,5000,5000
1,1,Feb,5000,10000
2,1,Mar,5000,15000
3,1,Apr,5000,20000
4,1,May,5000,25000
5,1,Jun,5000,30000
6,1,Jul,5000,35000
7,2,Jan,7000,7000
8,2,Feb,7000,14000
9,2,Mar,7000,21000


**Explanation**

- `PARTITION BY emp_id` resets the cumulative sum for each employee separately.
- Since months are stored as text (Jan, Feb...) we use a `CASE` statement to convert them to numbers (1–7) for correct ordering.
- `SUM(salary) OVER (...)` accumulates the salary month by month within each employee.
- Without the `CASE` ordering the months would sort alphabetically — Apr, Feb, Jan... which is wrong.

## 7. Tables X and Y — Row Count for != Join Types

**Difficulty:** 🔴 Hard

**Subtopic:** JOIN with != condition + Cartesian Product logic

---

**Tables: table_x + table_y**
```
table_x          table_y
+--------+        +--------+
| ids    |        | ids    |
+--------+        +--------+
| int    |        | int    |
+--------+        +--------+
```

---

**Problem**

Given:

table_x — 4 rows all with value 1:
```
+-----+
| ids |
+-----+
| 1   |
| 1   |
| 1   |
| 1   |
+-----+
```

table_y — 8 rows all with value 1:
```
+-----+
| ids |
+-----+
| 1   |
| 1   |
| 1   |
| 1   |
| 1   |
| 1   |
| 1   |
| 1   |
+-----+
```

Find the row count for:
- `SELECT * FROM X JOIN Y ON X.ids != Y.ids`
- `SELECT * FROM X LEFT JOIN Y ON X.ids != Y.ids`
- `SELECT * FROM X RIGHT JOIN Y ON X.ids != Y.ids`
- `SELECT * FROM X FULL OUTER JOIN Y ON X.ids != Y.ids`

**Expected Output**
```
+------------------+-----------+
| join_type        | row_count |
+------------------+-----------+
| != INNER JOIN    | 0         |
| != LEFT JOIN     | 4         |
| != RIGHT JOIN    | 8         |
| != FULL OUTER    | 12        |
+------------------+-----------+
```

✅ != INNER JOIN

In [18]:
# ✅ != INNER JOIN
print("!= INNER JOIN:")
pd.read_sql_query("""
SELECT COUNT(*) AS row_count
FROM table_x x
JOIN table_y y ON x.ids != y.ids
""", conn)

!= INNER JOIN:


,row_count
0,0


 LEFT JOIN

In [19]:
# ✅ != LEFT JOIN
print("!= LEFT JOIN:")
pd.read_sql_query("""
SELECT COUNT(*) AS row_count
FROM table_x x
LEFT JOIN table_y y ON x.ids != y.ids
""", conn)

!= LEFT JOIN:


,row_count
0,4


RIGHT JOIN

In [20]:
# ✅ != RIGHT JOIN (simulated)
print("!= RIGHT JOIN:")
pd.read_sql_query("""
SELECT COUNT(*) AS row_count
FROM table_y y
LEFT JOIN table_x x ON x.ids != y.ids
""", conn)

!= RIGHT JOIN:


,row_count
0,8


FULL OUTER JOIN

In [21]:
# ✅ != FULL OUTER JOIN (simulated)
print("!= FULL OUTER JOIN:")
pd.read_sql_query("""
SELECT COUNT(*) AS row_count FROM (
    SELECT x.ids FROM table_x x
    LEFT JOIN table_y y ON x.ids != y.ids
    UNION ALL
    SELECT y.ids FROM table_y y
    LEFT JOIN table_x x ON x.ids != y.ids
    WHERE x.ids IS NULL
)
""", conn)

!= FULL OUTER JOIN:


,row_count
0,12


**Explanation**

- Since **all values in both tables are 1**, the condition `X.ids != Y.ids` is **never true**.
- **!= INNER JOIN** — no rows match the condition → **0 rows**
- **!= LEFT JOIN** — no matches found so all 4 rows from X are returned with NULLs from Y → **4 rows**
- **!= RIGHT JOIN** — no matches found so all 8 rows from Y are returned with NULLs from X → **8 rows**
- **!= FULL OUTER JOIN** — all unmatched rows from both sides → 4 + 8 = **12 rows**

> 💡 Key insight — when the JOIN condition is never satisfied:
> - INNER JOIN = 0 rows always
> - LEFT JOIN = rows from left table only
> - RIGHT JOIN = rows from right table only
> - FULL OUTER = rows from both tables combined

## 8. Zomato — Multi-table Analysis

**Difficulty:** 🟡 Medium

**Subtopic:** GROUP BY + Aggregation + JOIN

---

**Table: zomato_orders**
```
+---------------------+---------+
| Column Name         | Type    |
+---------------------+---------+
| user_id             | int     |
| order_id            | int     |
| amount              | int     |
| order_date          | date    |
| delivery_partner_id | int     |
| delivery_rating     | int     |
+---------------------+---------+
```

- `user_id` represents the user who placed the order.
- `amount` represents the order amount.
- `delivery_partner_id` represents the delivery partner.
- `delivery_rating` represents the rating given to the delivery partner.

---

**Problem**

Using the zomato_orders table answer the following:

1. Calculate the **total amount spent** by each user.
2. Calculate the **average delivery rating** for each partner.
3. Find the **highest and lowest order amount** for each user.

**Example 1**

**Input**

zomato_orders table:
```
+---------+----------+--------+------------+---------------------+-----------------+
| user_id | order_id | amount | order_date | delivery_partner_id | delivery_rating |
+---------+----------+--------+------------+---------------------+-----------------+
| 1       | 101      | 200    | 2024-01-01 | 1                   | 5               |
| 1       | 102      | 300    | 2024-01-03 | 2                   | 4               |
| 2       | 103      | 150    | 2024-01-04 | 1                   | 5               |
| 3       | 104      | 400    | 2024-01-05 | 3                   | 3               |
+---------+----------+--------+------------+---------------------+-----------------+
```

**Output 1 — Total amount per user:**
```
+---------+--------------+
| user_id | total_amount |
+---------+--------------+
| 1       | 500          |
| 2       | 150          |
| 3       | 400          |
+---------+--------------+
```

**Output 2 — Avg delivery rating per partner:**
```
+---------------------+----------------+
| delivery_partner_id | avg_rating     |
+---------------------+----------------+
| 1                   | 5.0            |
| 2                   | 4.0            |
| 3                   | 3.0            |
+---------------------+----------------+
```

**Output 3 — Highest and lowest order per user:**
```
+---------+------------+------------+
| user_id | max_amount | min_amount |
+---------+------------+------------+
| 1       | 300        | 200        |
| 2       | 150        | 150        |
| 3       | 400        | 400        |
+---------+------------+------------+
```

In [22]:
# 👀 Preview the table
print("zomato_orders table:")
pd.read_sql_query("SELECT * FROM zomato_orders", conn)

zomato_orders table:


,user_id,order_id,amount,order_date,delivery_partner_id,delivery_rating
0,1,101,200,2024-01-01,1,5
1,1,102,300,2024-01-03,2,4
2,2,103,150,2024-01-04,1,5
3,3,104,400,2024-01-05,3,3


In [23]:
# ✅ Query 1 — Total amount spent per user
print("1. Total amount spent per user:")
pd.read_sql_query("""
SELECT user_id,
       SUM(amount) AS total_amount
FROM zomato_orders
GROUP BY user_id
""", conn)

1. Total amount spent per user:


,user_id,total_amount
0,1,500
1,2,150
2,3,400


In [24]:
# ✅ Query 2 — Average delivery rating per partner
print("2. Average delivery rating per partner:")
pd.read_sql_query("""
SELECT delivery_partner_id,
       AVG(delivery_rating) AS avg_rating
FROM zomato_orders
GROUP BY delivery_partner_id
""", conn)

2. Average delivery rating per partner:


,delivery_partner_id,avg_rating
0,1,5.0
1,2,4.0
2,3,3.0


In [25]:
# ✅ Query 3 — Highest and lowest order amount per user
print("3. Highest and lowest order amount per user:")
pd.read_sql_query("""
SELECT user_id,
       MAX(amount) AS max_amount,
       MIN(amount) AS min_amount
FROM zomato_orders
GROUP BY user_id
""", conn)

3. Highest and lowest order amount per user:


,user_id,max_amount,min_amount
0,1,300,200
1,2,150,150
2,3,400,400


**Explanation**

**Query 1 — Total amount per user:**
- `GROUP BY user_id` groups all orders by each user.
- `SUM(amount)` adds up all order amounts for each user.

**Query 2 — Avg delivery rating per partner:**
- `GROUP BY delivery_partner_id` groups by each delivery partner.
- `AVG(delivery_rating)` calculates the average rating received by each partner.

**Query 3 — Highest and lowest per user:**
- `MAX(amount)` finds the single largest order for each user.
- `MIN(amount)` finds the single smallest order for each user.
- Both aggregations happen within the same `GROUP BY user_id`.